# Solutions · 03 · Variance reduction

Worked solutions to the exercises of [notebook 03](../notebooks/03_variance_reduction.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engstoch is missing): install it from GitHub
    import engstoch
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engstoch"], check=True)
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from engstoch import montecarlo as mc, brownian as bm, special as sf
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

R = np.random.default_rng      # R(seed): a fresh, reproducible generator

## Exercise 1

Price an Asian (arithmetic-average) call on GBM with 12 monitoring dates, using the geometric-average Asian call (closed form: a lognormal) as a control variate. Report the variance reduction.

In [2]:
S0, K, r, sig, T, m = 100.0, 100.0, 0.05, 0.2, 1.0, 12
n = 100_000
S = bm.gbm(S0, r, sig, T, m, n, R(1))[:, 1:]
arith = np.exp(-r * T) * np.maximum(S.mean(axis=1) - K, 0)
geo = np.exp(-r * T) * np.maximum(np.exp(np.log(S).mean(axis=1)) - K, 0)
t = np.arange(1, m + 1) * T / m
mu_g = np.log(S0) + (r - sig**2 / 2) * t.mean()
var_g = sig**2 / m**2 * np.sum(np.minimum.outer(t, t))
d1 = (mu_g - np.log(K) + var_g) / np.sqrt(var_g)
geo_exact = np.exp(-r * T) * (np.exp(mu_g + var_g / 2) * sf.norm_cdf(d1) - K * sf.norm_cdf(d1 - np.sqrt(var_g)))
plain = mc.mean_ci(arith)
cv = mc.control_variate(arith, geo, geo_exact)
print(f"geometric Asian (closed form) {geo_exact:.4f}, simulated {geo.mean():.4f}")
print(f"arithmetic Asian: plain {plain['estimate']:.4f} +- {plain['std_error']:.4f}; with the control {cv['estimate']:.4f} +- {cv['std_error']:.5f}; variance reduction {cv['variance_reduction']:.0f}x")

geometric Asian (closed form) 5.9402, simulated 5.9111
arithmetic Asian: plain 6.1258 +- 0.0267; with the control 6.1558 +- 0.00074; variance reduction 1318x


The log of the geometric average of GBM prices is normal (a linear combination of Brownian values) with mean
log S₀ + (r − σ²/2)t̄ and variance σ²/m² Σᵢⱼ min(tᵢ, tⱼ), so its call has a Black-Scholes-type formula. The two
averages are almost perfectly correlated, and the control variate cuts the variance by several hundred - the
classical Kemna-Vorst trick.

## Exercise 2

Estimate P(S_100 > 30) for a sum of 100 Exp(1) variables minus 100 (i.e. P(sum > 130)) by exponential tilting of each summand. Find the best tilt and compare with the gamma tail from SciPy.

In [3]:
n, a = 100, 130.0
exact = stats.gamma.sf(a, n)
rows = []
for theta in (0.0, 0.1, 0.2, 0.23, 0.3, 0.4):
    g = R(int(100 * theta) + 1)
    X = g.exponential(1 / (1 - theta), (50_000, n))          # tilted Exp(1): rate 1 - theta
    S = X.sum(axis=1)
    lr = np.exp(-theta * S) * (1 - theta) ** (-n)              # f/g = prod e^{-x} / ((1-theta) e^{-(1-theta) x})
    est = mc.mean_ci((S > a) * lr)
    rows.append((theta, est["estimate"], mc.relative_error(est) if est["estimate"] > 0 else np.inf))
print(f"P(sum of 100 Exp(1) > 130) = {exact:.4e}")
print(pd.DataFrame(rows, columns=["tilt theta", "estimate", "relative error"]).to_string(index=False, float_format="%.3g"))
print("theory: the tilted mean n/(1 - theta) equals a at theta = 1 - n/a =", round(1 - n / a, 4))

P(sum of 100 Exp(1) > 130) = 2.7504e-03
 tilt theta  estimate  relative error
          0   0.00258          0.0879
        0.1    0.0028          0.0206
        0.2   0.00276         0.00917
       0.23   0.00272         0.00846
        0.3   0.00272          0.0101
        0.4   0.00272          0.0338
theory: the tilted mean n/(1 - theta) equals a at theta = 1 - n/a = 0.2308


Tilting each Exp(1) summand to Exp(1 − θ) multiplies the density by e^{θx}(1 − θ); the likelihood ratio of the
whole sample depends only on the sum. The best tilt makes the rare event typical - the tilted mean of the sum,
n/(1 − θ), equals the threshold 130 at θ = 0.23 - and gives about 1 % relative error with 5·10⁴ samples, where
crude Monte Carlo would need around 10⁸.

## Exercise 3

**Implement it yourself:** write stratified sampling with Neyman allocation for E[h(U)] from scratch (pilot run, allocation, estimate and standard error), and reproduce the library's result for h(u) = 10u on u > 0.9.

In [4]:
h = lambda u: np.where(u > 0.9, 10 * u, 0.1 * u)
def stratified_neyman(h, n, k, g, pilot=200):
    edges = np.linspace(0, 1, k + 1); p = np.diff(edges)
    sd = np.array([np.std(h(edges[j] + p[j] * g.random(pilot)), ddof=1) for j in range(k)])
    nj = np.maximum(2, np.round(n * p * sd / np.sum(p * sd)).astype(int))
    m, v = np.zeros(k), np.zeros(k)
    for j in range(k):
        y = h(edges[j] + p[j] * g.random(nj[j])); m[j], v[j] = y.mean(), y.var(ddof=1)
    return np.sum(p * m), np.sqrt(np.sum(p**2 * v / nj)), nj
est, se, nj = stratified_neyman(h, 10_000, 10, R(5))
lib = mc.stratified(h, 10_000, 10, R(5), "neyman")
print(f"mine: {est:.5f} +- {se:.5f}, allocation {nj.tolist()}\nlibrary: {lib['estimate']:.5f} +- {lib['std_error']:.5f}, allocation {lib['allocation'].tolist()}\nexact {0.1 * 0.81 / 2 + 10 * 0.19 / 2:.5f}")

mine: 0.99052 +- 0.00031, allocation [93, 98, 93, 95, 98, 99, 86, 97, 93, 9149]
library: 0.99052 +- 0.00031, allocation [93, 98, 93, 95, 98, 99, 86, 97, 93, 9149]
exact 0.99050


The pilot estimates each stratum's standard deviation; Neyman allocation puts most samples in the last
stratum, where the integrand jumps. With the same random stream the hand-written version reproduces the
library's allocation and estimate exactly.